In [ ]:
from nn2 import *
i = 64
Rms = lambda*_:Res(RMSNorm(i), *_)
CPA = lambda:(
    Rms(Add(
        Cnv(i, g=1., r=(3,3)),
        Pol(i, h=.25),
        At(i, h=.25, G=1, g=4, r=(12,11)),
        Swi(i, h=2.),
    )),
    Rms(Add(
        Cnv(i, g=1., r=(3,3)),
        Pol(i, h=.25),
        At(i, h=.25, G=1, g=4, r=(12,11)),
        Swi(i, h=2.),
    )),
    Rms(Add(
        Cnv(i, g=1., r=(3,3)),
        Pol(i, h=.25),
        At(i, h=.25, G=1, g=4, r=(12,11)),
        Swi(i, h=2.),
    )),
)
mdl = Seq(
    Zip(
        Embedding(256, i),
        Linear(i, i, 0),
        Linear(i, i, 0)
    ),
    Seq(
        Rms(Swi(i, h=2.)),
        *(_:=lambda n:[
            *CPA(),
            Rms(
                Linear(i, i, 0),
                T(MaxPool1d(2)),
                Linear(i, i, 0),
                *_(n-1),
                Linear(i, i*2),
                Unflatten(2, (2,-1)),
                Flatten(1, 2)
            ),
            *CPA()
        ]if n else CPA())(7)
    ),
    Swi(i, 2., 256),
).to('cuda',torch.bfloat16)
mdl[1] = torch.compile(mdl[1], fullgraph=True, dynamic=False)
opt = torch.optim.AdamW(mdl.parameters(), fused=True)
lr = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, 100000, cycle_momentum=False)
plot = Plot()
for x,t in Dl():
    if lr.last_epoch == lr.total_steps: break
    opt.zero_grad()
    x = mdl[:2](x)
    y = mdl[2](x),
    y += mdl([y[0].argmax(2), x, y[0].softmax(2)@mdl[0][0].weight]),
    l = [functional.cross_entropy(y.mT,t)for y in y]
    reduce(torch.add,l).backward()
    opt.step()
    lr.step()
    plot(l=torch.stack(l).data, a=torch.stack([(y.argmax(2)==t).sum()/t.numel()for y in y]))

In [ ]:
class LogRes(Res):
    def forward(s, _):
        __ = super().forward(_)
        s.log = __.norm()/_.norm()
        return __
for _ in mdl.modules():
    if isinstance(_, Res):
        _.__class__ = LogRes
with torch.no_grad():
    mdl[2](mdl[1]._orig_mod(mdl[0](next(Dl())[0])))
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],Cnv)]).to('cpu',torch.float), label='cnv')
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],Pol)]).to('cpu',torch.float), label='pol')
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],At)]).to('cpu',torch.float), label='at')
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],Swi)]).to('cpu',torch.float), label='swi')
plt.legend()

In [ ]:
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and type(_[1])==Linear]).to('cpu',torch.float));

In [ ]:
[i for i in dir(torch) if 'cum' in i.lower() and 'avg' in i.lower()]

In [ ]:
from nn2 import *
i = 64
Rms = lambda*_:Res(RMSNorm(i), *_)
CPA = lambda:(
    
    Rms(Cnv(i, g=1., r=(1,1))),
    Rms(Pol(i, h=.25)),
    Rms(At(i, h=.25, G=1, g=4, r=(12,11))),
    Rms(Swi(i, h=2.)),
    Rms(Cnv(i, g=1., r=(1,1))),
    Rms(Pol(i, h=.25)),
    Rms(At(i, h=.25, G=1, g=4, r=(12,11))),
    Rms(Swi(i, h=2.)),
)
mdl = Seq(
    Zip(
        Embedding(256, i),
        Linear(i, i, 0),
        Linear(i, i, 0)
    ),
    Seq(*(_:=lambda n:[
        *CPA(),
        Rms(
            Linear(i, i, 0),
            T(MaxPool1d(2)),
            Linear(i, i, 0),
            *_(n-1),
            Linear(i, i*2),
            Unflatten(2, (2,-1)),
            Flatten(1, 2)
        ),
        *CPA()
    ]if n else CPA())(7)),
    Swi(i, 2., 256),
).to('cuda',torch.bfloat16)
mdl[1] = torch.compile(mdl[1], fullgraph=True, dynamic=False)
opt = torch.optim.Adam(mdl.parameters(), fused=True)
lr = torch.optim.lr_scheduler.OneCycleLR(opt, 5e-3, 200000, cycle_momentum=False)
plot = Plot()
for x,t in Dl():
    if lr.last_epoch == lr.total_steps: break
    opt.zero_grad()
    x = mdl[:2](x)
    y = mdl[2](x),
    y += mdl([y[0].argmax(2), x, y[0].softmax(2)@mdl[0][0].weight]),
    l = [functional.cross_entropy(y.mT,t)for y in y]
    reduce(torch.add,l).backward()
    opt.step()
    lr.step()
    plot(l=torch.stack(l).data, a=torch.stack([(y.argmax(2)==t).sum()/t.numel()for y in y]))

In [ ]:
l

In [ ]:
class LogRes(Res):
    def forward(s, _):
        __ = super().forward(_)
        s.log = __.norm()/_.norm()
        return __
for _ in mdl.modules():
    if isinstance(_, Res):
        _.__class__ = LogRes
with torch.no_grad():
    mdl[2](mdl[1]._orig_mod(mdl[0](next(Dl())[0])))
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],Cnv)]).to('cpu',torch.float), label='cnv')
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],Pol)]).to('cpu',torch.float), label='pol')
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],At)]).to('cpu',torch.float), label='at')
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and isinstance(_[1],Swi)]).to('cpu',torch.float), label='swi')
plt.legend()

In [ ]:
plt.plot(torch.stack([_.log for _ in mdl.modules()if isinstance(_,Res)and type(_[1])==Linear]).to('cpu',torch.float));